# Crime Data Pipeline — Police Force Analytics Unit
**Forces:** Metropolitan Police, West Midlands Police, West Yorkshire Police, Thames Valley Police  
**Period:** April 2023 – March 2026  
**Author:** Izzam Golamaully
**Date:** May 2026  

## Pipeline Stages
1. Ingestion Layer
2. Cleaning & Validation Layer
3. Feature Engineering & Transformation
4. Enrichment & Aggregation
5. Export & Final Validation

In [12]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print("Libraries loaded ✓")

Libraries loaded ✓


In [13]:
# All paths relative to the notebook location
BASE_DIR = Path("..") 
RAW_POLICE_DIR = BASE_DIR / "data" / "raw" / "police"
ENRICHMENT_DIR = BASE_DIR / "data" / "raw" / "enrichment"
OUTPUT_DIR     = BASE_DIR / "data" / "outputs"

# Create output dir if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths exist
for p in [RAW_POLICE_DIR, ENRICHMENT_DIR]:
    status = "✓" if p.exists() else "✗ NOT FOUND"
    print(f"{p}: {status}")

..\data\raw\police: ✓
..\data\raw\enrichment: ✓


In [14]:
# Get all CSV files in the police data folder
all_files = glob.glob(str(RAW_POLICE_DIR / "**" / "*.csv"), recursive=True)

print(f"Total CSV files found: {len(all_files)}")
print("\nFirst 10 files:")
for f in sorted(all_files)[:10]:
    print(f)

Total CSV files found: 144

First 10 files:
..\data\raw\police\2023-04\2023-04-metropolitan-street.csv
..\data\raw\police\2023-04\2023-04-thames-valley-street.csv
..\data\raw\police\2023-04\2023-04-west-midlands-street.csv
..\data\raw\police\2023-04\2023-04-west-yorkshire-street.csv
..\data\raw\police\2023-05\2023-05-metropolitan-street.csv
..\data\raw\police\2023-05\2023-05-thames-valley-street.csv
..\data\raw\police\2023-05\2023-05-west-midlands-street.csv
..\data\raw\police\2023-05\2023-05-west-yorkshire-street.csv
..\data\raw\police\2023-06\2023-06-metropolitan-street.csv
..\data\raw\police\2023-06\2023-06-thames-valley-street.csv


In [15]:
# Load just one file to inspect structure
sample_file = sorted(all_files)[0]
print(f"Inspecting: {sample_file}\n")

sample_df = pd.read_csv(sample_file)

print(f"Shape: {sample_df.shape}")
print(f"\nColumns:\n{list(sample_df.columns)}")
print(f"\nData types:\n{sample_df.dtypes}")
print(f"\nFirst 3 rows:")
sample_df.head(3)

Inspecting: ..\data\raw\police\2023-04\2023-04-metropolitan-street.csv

Shape: (87547, 12)

Columns:
['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']

Data types:
Crime ID                  object
Month                     object
Reported by               object
Falls within              object
Longitude                float64
Latitude                 float64
Location                  object
LSOA code                 object
LSOA name                 object
Crime type                object
Last outcome category     object
Context                  float64
dtype: object

First 3 rows:


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,7c85d5925620a55fd890ea37dd748a34d2c77773f5d740...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.254186,50.834677,On or near Kingsland Close,E01031372,Adur 004E,Vehicle crime,Investigation complete; no suspect identified,NaN
1,43ccf73d9d380e4fcd128d7384d49ccf8d5ed54887a940...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.559343,50.852819,On or near School Lane,E01031392,Arun 001C,Violence and sexual offences,Investigation complete; no suspect identified,NaN
2,dd453aa29e0058e3fd98c507bf9429b10d654b653571f7...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.405658,50.865992,On or near Southview Road,E01031425,Arun 002C,Violence and sexual offences,Investigation complete; no suspect identified,NaN


In [16]:
# Check null % on the sample file before ingesting everything
print("Null % per column:")
null_pct = (sample_df.isnull().sum() / len(sample_df) * 100).round(2)
print(null_pct)

print(f"\nUnique crime types ({sample_df['Crime type'].nunique()}):")
print(sorted(sample_df['Crime type'].unique()))

Null % per column:
Crime ID                  19.80
Month                      0.00
Reported by                0.00
Falls within               0.00
Longitude                  2.31
Latitude                   2.31
Location                   0.00
LSOA code                  2.31
LSOA name                  2.31
Crime type                 0.00
Last outcome category     19.80
Context                  100.00
dtype: float64

Unique crime types (14):
['Anti-social behaviour', 'Bicycle theft', 'Burglary', 'Criminal damage and arson', 'Drugs', 'Other crime', 'Other theft', 'Possession of weapons', 'Public order', 'Robbery', 'Shoplifting', 'Theft from the person', 'Vehicle crime', 'Violence and sexual offences']


---
## Stage 1: Ingestion Layer

Batch ingestion — one force at a time. Each CSV is loaded individually and 
unioned into a single DataFrame. The `force_name` field is extracted from 
the filename. Raw row counts are logged per force for validation.

In [17]:
# Force name mapping — extracted from filename to standardised label
FORCE_MAP = {
    "metropolitan": "Metropolitan Police",
    "west-midlands": "West Midlands Police",
    "west-yorkshire": "West Yorkshire Police",
    "thames-valley": "Thames Valley Police"
}

# Columns to keep from raw data (drop Context — 100% null)
KEEP_COLS = [
    "Crime ID", "Month", "Longitude", "Latitude",
    "LSOA code", "LSOA name", "Crime type", "Last outcome category"
]

def extract_force_name(filepath):
    """Extract standardised force name from filename."""
    filename = Path(filepath).stem  # e.g. '2023-04-metropolitan-street'
    for key, name in FORCE_MAP.items():
        if key in filename:
            return name
    return "Unknown"

def ingest_force_files(file_list):
    """
    Load a list of CSV files, add force_name column,
    keep only required columns, and return combined DataFrame.
    """
    chunks = []
    for fp in file_list:
        df = pd.read_csv(fp, usecols=KEEP_COLS)
        df["force_name"] = extract_force_name(fp)
        df["source_file"] = Path(fp).name
        chunks.append(df)
    return pd.concat(chunks, ignore_index=True)


# ── Ingest by force (one force at a time) ──────────────────────────────────
print("=" * 55)
print("INGESTION LAYER — Row counts per force")
print("=" * 55)

raw_dfs = []
total_files = 0

for key, force_name in FORCE_MAP.items():
    force_files = [f for f in all_files if key in Path(f).stem]
    force_df = ingest_force_files(force_files)
    raw_dfs.append(force_df)
    total_files += len(force_files)
    print(f"{force_name:<30} | {len(force_files):>3} files | {len(force_df):>8,} rows")

# Union all forces
raw_df = pd.concat(raw_dfs, ignore_index=True)

print("-" * 55)
print(f"{'TOTAL':<30} | {total_files:>3} files | {len(raw_df):>8,} rows")
print(f"\nShape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")

INGESTION LAYER — Row counts per force
Metropolitan Police            |  36 files | 3,415,820 rows
West Midlands Police           |  36 files | 1,011,509 rows
West Yorkshire Police          |  36 files |  926,624 rows
Thames Valley Police           |  36 files |  593,125 rows
-------------------------------------------------------
TOTAL                          | 144 files | 5,947,078 rows

Shape: (5947078, 10)
Columns: ['Crime ID', 'Month', 'Longitude', 'Latitude', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'force_name', 'source_file']


---
## Stage 2: Cleaning & Validation Layer

Data quality assessment, null handling, duplicate detection, 
and validation checks. Row counts logged before and after 
each transformation.

In [18]:
print("=" * 55)
print("DATA QUALITY ASSESSMENT — Raw ingested data")
print("=" * 55)

print(f"\nTotal rows: {len(raw_df):,}")
print(f"Total columns: {len(raw_df.columns)}")

print("\nNull % per column:")
null_pct = (raw_df.isnull().sum() / len(raw_df) * 100).round(2)
for col, pct in null_pct.items():
    print(f"  {col:<30} {pct:>6}%")

print(f"\nDuplicate Crime IDs (excl. nulls):")
crime_id_dupes = raw_df[raw_df['Crime ID'].notna()].duplicated(subset='Crime ID').sum()
print(f"  {crime_id_dupes:,} duplicate Crime IDs")

print(f"\nUnique forces: {raw_df['force_name'].nunique()}")
print(f"Forces present: {sorted(raw_df['force_name'].unique())}")

print(f"\nDate range:")
print(f"  Earliest month: {raw_df['Month'].min()}")
print(f"  Latest month:   {raw_df['Month'].max()}")
print(f"  Unique months:  {raw_df['Month'].nunique()}")

print(f"\nUnique crime types: {raw_df['Crime type'].nunique()}")
print(sorted(raw_df['Crime type'].unique()))

DATA QUALITY ASSESSMENT — Raw ingested data

Total rows: 5,947,078
Total columns: 10

Null % per column:
  Crime ID                        15.34%
  Month                             0.0%
  Longitude                        0.66%
  Latitude                         0.66%
  LSOA code                        0.66%
  LSOA name                        0.66%
  Crime type                        0.0%
  Last outcome category           15.34%
  force_name                        0.0%
  source_file                       0.0%

Duplicate Crime IDs (excl. nulls):
  38,636 duplicate Crime IDs

Unique forces: 4
Forces present: ['Metropolitan Police', 'Thames Valley Police', 'West Midlands Police', 'West Yorkshire Police']

Date range:
  Earliest month: 2023-04
  Latest month:   2026-03
  Unique months:  36

Unique crime types: 14
['Anti-social behaviour', 'Bicycle theft', 'Burglary', 'Criminal damage and arson', 'Drugs', 'Other crime', 'Other theft', 'Possession of weapons', 'Public order', 'Robbery', 'Sho

In [19]:
rows_before = len(raw_df)
print(f"Rows before cleaning: {rows_before:,}")

# ── 1. Rename columns to snake_case ────────────────────────────────────────
clean_df = raw_df.rename(columns={
    "Crime ID":               "crime_id",
    "Month":                  "year_month",
    "Longitude":              "longitude",
    "Latitude":               "latitude",
    "LSOA code":              "lsoa_code",
    "LSOA name":              "lsoa_name",
    "Crime type":             "crime_category",
    "Last outcome category":  "last_outcome",
})

# ── 2. Drop source_file (pipeline metadata, not needed downstream) ─────────
clean_df = clean_df.drop(columns=["source_file"])

# ── 3. Parse year_month to datetime ────────────────────────────────────────
clean_df["year_month"] = pd.to_datetime(clean_df["year_month"], format="%Y-%m")

# ── 4. Validate date range — drop anything outside Apr 2023 – Mar 2026 ─────
valid_start = pd.Timestamp("2023-04-01")
valid_end   = pd.Timestamp("2026-03-01")

out_of_range = clean_df[
    (clean_df["year_month"] < valid_start) | 
    (clean_df["year_month"] > valid_end)
]
print(f"\nOut-of-range dates found: {len(out_of_range):,}")
clean_df = clean_df[
    (clean_df["year_month"] >= valid_start) & 
    (clean_df["year_month"] <= valid_end)
]

# ── 5. Strip whitespace from string columns ────────────────────────────────
str_cols = ["crime_id", "lsoa_code", "lsoa_name", "crime_category", 
            "last_outcome", "force_name"]
for col in str_cols:
    clean_df[col] = clean_df[col].str.strip()

# ── 6. Standardise crime_category casing ──────────────────────────────────
clean_df["crime_category"] = clean_df["crime_category"].str.title()

# ── 7. Flag location-suppressed records (don't drop — still useful) ────────
clean_df["location_suppressed"] = clean_df["longitude"].isnull().astype(int)

# ── Validation summary ─────────────────────────────────────────────────────
rows_after = len(clean_df)
print(f"Rows after cleaning:  {rows_after:,}")
print(f"Rows removed:         {rows_before - rows_after:,}")
print(f"\nNull % after cleaning:")
null_pct = (clean_df.isnull().sum() / len(clean_df) * 100).round(2)
for col, pct in null_pct.items():
    flag = " ← expected (ASB records)" if pct > 10 else ""
    print(f"  {col:<30} {pct:>6}%{flag}")

print(f"\nCrime categories after standardisation:")
print(sorted(clean_df["crime_category"].unique()))

print(f"\nSample cleaned row:")
clean_df.head(1)

Rows before cleaning: 5,947,078

Out-of-range dates found: 0
Rows after cleaning:  5,947,078
Rows removed:         0

Null % after cleaning:
  crime_id                        15.34% ← expected (ASB records)
  year_month                        0.0%
  longitude                        0.66%
  latitude                         0.66%
  lsoa_code                        0.66%
  lsoa_name                        0.66%
  crime_category                    0.0%
  last_outcome                    15.34% ← expected (ASB records)
  force_name                        0.0%
  location_suppressed               0.0%

Crime categories after standardisation:
['Anti-Social Behaviour', 'Bicycle Theft', 'Burglary', 'Criminal Damage And Arson', 'Drugs', 'Other Crime', 'Other Theft', 'Possession Of Weapons', 'Public Order', 'Robbery', 'Shoplifting', 'Theft From The Person', 'Vehicle Crime', 'Violence And Sexual Offences']

Sample cleaned row:


,crime_id,year_month,longitude,latitude,lsoa_code,lsoa_name,crime_category,last_outcome,force_name,location_suppressed
0,7c85d5925620a55fd890ea37dd748a34d2c77773f5d740...,2023-04-01,-0.254186,50.834677,E01031372,Adur 004E,Vehicle Crime,Investigation complete; no suspect identified,Metropolitan Police,0


In [20]:
# Standardise to clean, consistent labels
CATEGORY_MAP = {
    "Anti-Social Behaviour":        "Anti-Social Behaviour",
    "Bicycle Theft":                "Bicycle Theft",
    "Burglary":                     "Burglary",
    "Criminal Damage And Arson":    "Criminal Damage and Arson",
    "Drugs":                        "Drug Offences",
    "Other Crime":                  "Other Crime",
    "Other Theft":                  "Other Theft",
    "Possession Of Weapons":        "Possession of Weapons",
    "Public Order":                 "Public Order",
    "Robbery":                      "Robbery",
    "Shoplifting":                  "Shoplifting",
    "Theft From The Person":        "Theft from the Person",
    "Vehicle Crime":                "Vehicle Crime",
    "Violence And Sexual Offences": "Violence and Sexual Offences",
}

clean_df["crime_category"] = clean_df["crime_category"].map(CATEGORY_MAP)

# Validate — check nothing got lost
unmapped = clean_df["crime_category"].isnull().sum()
print(f"Unmapped categories: {unmapped}")
print(f"\nFinal crime categories:")
print(sorted(clean_df["crime_category"].unique()))

Unmapped categories: 0

Final crime categories:
['Anti-Social Behaviour', 'Bicycle Theft', 'Burglary', 'Criminal Damage and Arson', 'Drug Offences', 'Other Crime', 'Other Theft', 'Possession of Weapons', 'Public Order', 'Robbery', 'Shoplifting', 'Theft from the Person', 'Vehicle Crime', 'Violence and Sexual Offences']


---
## Stage 3: Feature Engineering & Transformation

Deriving time attributes, adding severity weights, and preparing 
fields needed for aggregation and BI reporting.

In [21]:
# ── 1. Time attributes ─────────────────────────────────────────────────────
clean_df["year"]      = clean_df["year_month"].dt.year
clean_df["month_num"] = clean_df["year_month"].dt.month
clean_df["month_name"]= clean_df["year_month"].dt.strftime("%b")  # e.g. "Apr"

# ── 2. Crime severity weights (Cambridge Crime Harm Index) ─────────────────
SEVERITY_MAP = {
    "Violence and Sexual Offences": 36,
    "Robbery":                      18,
    "Burglary":                     8,
    "Drug Offences":                3,
    "Vehicle Crime":                4,
    "Bicycle Theft":                2,
    "Criminal Damage and Arson":    2,
    "Other Theft":                  2,
    "Theft from the Person":        2,
    "Shoplifting":                  1,
    "Public Order":                 2,
    "Possession of Weapons":        6,
    "Other Crime":                  2,
    "Anti-Social Behaviour":        1,  # Not in CHI — nominal weight
}

clean_df["severity_weight"] = clean_df["crime_category"].map(SEVERITY_MAP)

# ── Validation ─────────────────────────────────────────────────────────────
print("Feature engineering complete.")
print(f"\nNew columns added: year, month_num, month_name, severity_weight")
print(f"\nSeverity weight null check: {clean_df['severity_weight'].isnull().sum()}")
print(f"\nColumns now in clean_df:")
print(list(clean_df.columns))
print(f"\nSample row:")
clean_df[["force_name","year_month","year","month_num","crime_category","severity_weight"]].head(3)

Feature engineering complete.

New columns added: year, month_num, month_name, severity_weight

Severity weight null check: 0

Columns now in clean_df:
['crime_id', 'year_month', 'longitude', 'latitude', 'lsoa_code', 'lsoa_name', 'crime_category', 'last_outcome', 'force_name', 'location_suppressed', 'year', 'month_num', 'month_name', 'severity_weight']

Sample row:


,force_name,year_month,year,month_num,crime_category,severity_weight
0,Metropolitan Police,2023-04-01,2023,4,Vehicle Crime,4
1,Metropolitan Police,2023-04-01,2023,4,Violence and Sexual Offences,36
2,Metropolitan Police,2023-04-01,2023,4,Violence and Sexual Offences,36


In [23]:
enrichment_files = list(ENRICHMENT_DIR.glob("*"))
print(f"Files in enrichment folder: {len(enrichment_files)}")
for f in enrichment_files:
    print(f"  {f.name}")

Files in enrichment folder: 1
  pfatablesyedec2025.xlsx


In [24]:
import openpyxl

pop_file = ENRICHMENT_DIR / "pfatablesyedec2025.xlsx"

# List all sheet names
wb = openpyxl.load_workbook(pop_file, read_only=True)
print(f"Sheets in file ({len(wb.sheetnames)}):")
for sheet in wb.sheetnames:
    print(f"  {sheet}")
wb.close()

Sheets in file (18):
  Cover_sheet
  Table of Contents
  Notes- PFA 
  Table P1
  Table P2
  Table P3
  Table P4 
  Table P5 
  Table P6 
  Table P7 
  Table P8 
  Table P9
  Notes - CSP
  Table C1
  Table C2
  Table C3
  Table C4
  Table C5


In [25]:
toc = pd.read_excel(pop_file, sheet_name="Table of Contents", header=None)
print(toc.to_string())

                                                                                                                                               0                                                                                                                                                                                                       1
0                                                                                                                              Table of contents                                                                                                                                                                                                     NaN
1                                                                                                              This worksheet contains 2 tables.                                                                                                                                                                      

In [26]:
p3 = pd.read_excel(pop_file, sheet_name="Table P3", header=None)
print(p3.head(20).to_string())

                                                                                                                                                                                             0                           1                                                        2                                                          3                                                  4                            5         6                                           7                     8                        9                        10               11        12              13        14                               15                                              16                        17                18                     19             20           21                        22                         23             24                              25                     26                                    27
0                         Table P3:  Police recorded crime by offenc

In [27]:
# Read Table P3 properly, using row 7 as header
p3_full = pd.read_excel(pop_file, sheet_name="Table P3", header=7)

# Check column names
print("Columns:")
for i, col in enumerate(p3_full.columns):
    print(f"  {i}: {col}")

Columns:
  0: Area Code
  1: Area Name
  2: Population figures
 (mid-2024) rounded to 100 [note 4]
  3: Household figures
 (mid-2021)  rounded to 100
[note 5] 
  4: Total recorded crime (excluding fraud)
 [note 2]
  5: Violence against the person
  6: Homicide
  7: Death or serious injury - unlawful driving
  8: Violence with injury
  9: Violence without injury
  10: Stalking and harassment
  11: Sexual offences
  12: Robbery
  13: Theft offences
  14: Burglary
  15: Residential burglary
 [note 6]
  16: Residential burglary (households)
 [note 6,7]
  17: Non-residential burglary
  18: Vehicle offences
  19: Theft from the person
  20: Bicycle theft
  21: Shoplifting
  22: All other theft offences
  23: Criminal damage and arson
  24: Drug offences
  25: Possession of weapons offences
  26: Public order offences
  27: Miscellaneous crimes against society


In [28]:
# Rename columns we need
p3_full = p3_full.rename(columns={
    "Area Name": "force_name",
    p3_full.columns[2]: "population"
})

# Check what force names look like in this file
print("All force names in file:")
print(p3_full["force_name"].dropna().tolist())

All force names in file:
['ENGLAND AND WALES [note 7]', 'ENGLAND', 'North East', 'Cleveland', 'Durham', 'Northumbria', 'North West', 'Cheshire', 'Cumbria', 'Greater Manchester', 'Lancashire', 'Merseyside', 'Yorkshire and The Humber', 'Humberside', 'North Yorkshire', 'South Yorkshire', 'West Yorkshire', 'East Midlands', 'Derbyshire', 'Leicestershire', 'Lincolnshire', 'Northamptonshire', 'Nottinghamshire', 'West Midlands', 'Staffordshire', 'Warwickshire', 'West Mercia', 'West Midlands', 'East ', 'Bedfordshire', 'Cambridgeshire', 'Essex', 'Hertfordshire', 'Norfolk', 'Suffolk', 'London [note 9]', 'City of London', 'Metropolitan Police', 'South East', 'Hampshire', 'Kent', 'Surrey', 'Sussex', 'Thames Valley', 'South West', 'Avon and Somerset', 'Devon and Cornwall', 'Dorset', 'Gloucestershire', 'Wiltshire', 'WALES', 'Dyfed-Powys', 'Gwent', 'North Wales', 'South Wales']


In [29]:
# Map file names to our standardised force names
POPULATION_FORCE_MAP = {
    "Metropolitan Police": "Metropolitan Police",
    "West Midlands":       "West Midlands Police",
    "West Yorkshire":      "West Yorkshire Police",
    "Thames Valley":       "Thames Valley Police",
}

# Extract relevant rows and columns
pop_df = p3_full[p3_full["force_name"].isin(POPULATION_FORCE_MAP.keys())][["force_name", "population"]].copy()

# Apply standardised names
pop_df["force_name"] = pop_df["force_name"].map(POPULATION_FORCE_MAP)

# Add year column — mid-2024 figures applied to all years (documented assumption)
# We'll cross join to all years in our dataset later
pop_df["population"] = pop_df["population"].astype(int)

pop_df = pop_df.reset_index(drop=True)

print("Population data extracted:")
print(pop_df)
print(f"\nNote: mid-2024 ONS figures will be applied across all years (Apr 2023 – Mar 2026).")
print("This is documented as Assumption A1 in the Statement of Work.")

Population data extracted:
              force_name  population
0  West Yorkshire Police     2435200
1   West Midlands Police     6187200
2   West Midlands Police     3036600
3    Metropolitan Police     9074600
4   Thames Valley Police     2640200

Note: mid-2024 ONS figures will be applied across all years (Apr 2023 – Mar 2026).
This is documented as Assumption A1 in the Statement of Work.


In [30]:
# Check the duplicate — look at the raw rows in p3_full
wm_rows = p3_full[p3_full["force_name"] == "West Midlands"][["Area Code", "force_name", "population"]]
print(wm_rows)

    Area Code     force_name  population
23  E12000005  West Midlands     6187200
27  E23000014  West Midlands     3036600


In [31]:
# Re-extract population using Area Code to avoid the duplicate
FORCE_AREA_CODES = {
    "E23000001": "Metropolitan Police",   # Not in this file — use name match
    "E23000014": "West Midlands Police",
    "E23000004": "West Yorkshire Police", # Check this too
    "E23000043": "Thames Valley Police",  # Check this too
}

# Safer approach — filter by Area Code for West Midlands, name for others
pop_rows = []

# Metropolitan Police — match by name
met = p3_full[p3_full["force_name"] == "Metropolitan Police"][["force_name", "population"]].copy()
met["force_name"] = "Metropolitan Police"
pop_rows.append(met)

# West Midlands — match by Area Code E23000014
wm = p3_full[p3_full["Area Code"] == "E23000014"][["force_name", "population"]].copy()
wm["force_name"] = "West Midlands Police"
pop_rows.append(wm)

# West Yorkshire — match by name
wy = p3_full[p3_full["force_name"] == "West Yorkshire"][["force_name", "population"]].copy()
wy["force_name"] = "West Yorkshire Police"
pop_rows.append(wy)

# Thames Valley — match by name
tv = p3_full[p3_full["force_name"] == "Thames Valley"][["force_name", "population"]].copy()
tv["force_name"] = "Thames Valley Police"
pop_rows.append(tv)

# Combine
pop_df = pd.concat(pop_rows, ignore_index=True)
pop_df["population"] = pop_df["population"].astype(int)

print("Final population table:")
print(pop_df)
print(f"\nTotal forces: {pop_df['force_name'].nunique()}")

Final population table:
              force_name  population
0    Metropolitan Police     9074600
1   West Midlands Police     3036600
2  West Yorkshire Police     2435200
3   Thames Valley Police     2640200

Total forces: 4


In [32]:
imd_file = ENRICHMENT_DIR / "File_1_-_IMD2019_Index_of_Multiple_Deprivation.xlsx"

wb2 = openpyxl.load_workbook(imd_file, read_only=True)
print(f"Sheets ({len(wb2.sheetnames)}):")
for s in wb2.sheetnames:
    print(f"  {s}")
wb2.close()

Sheets (2):
  Notes
  IMD2019


In [33]:
imd_raw = pd.read_excel(imd_file, sheet_name="IMD2019")

print(f"Shape: {imd_raw.shape}")
print(f"\nColumns:")
print(list(imd_raw.columns))
print(f"\nFirst 3 rows:")
imd_raw.head(3)

Shape: (32844, 6)

Columns:
['LSOA code (2011)', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Rank', 'Index of Multiple Deprivation (IMD) Decile']

First 3 rows:


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Rank,Index of Multiple Deprivation (IMD) Decile
0,E01000001,City of London 001A,E09000001,City of London,29199,9
1,E01000002,City of London 001B,E09000001,City of London,30379,10
2,E01000003,City of London 001C,E09000001,City of London,14915,5


In [ ]:
# Rename columns to snake_case
imd_df = imd_raw.rename(columns={
    "LSOA code (2011)":                          "lsoa_code",
    "LSOA name (2011)":                          "lsoa_name",
    "Local Authority District code (2019)":      "lad_code",
    "Local Authority District name (2019)":      "lad_name",
    "Index of Multiple Deprivation (IMD) Rank":  "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile":"imd_decile",
})

print(f"Shape: {imd_df.shape}")
print(f"\nNull check:")
print(imd_df.isnull().sum())
print(f"\nIMD Decile range: {imd_df['imd_decile'].min()} – {imd_df['imd_decile'].max()}")
print(f"\nSample:")
imd_df.head(3)

Shape: (32844, 6)

Null check:
lsoa_code     0
lsoa_name     0
lad_code      0
lad_name      0
imd_rank      0
imd_decile    0
dtype: int64

IMD Decile range: 1 – 10

Sample:


,lsoa_code,lsoa_name,lad_code,lad_name,imd_rank,imd_decile
0,E01000001,City of London 001A,E09000001,City of London,29199,9
1,E01000002,City of London 001B,E09000001,City of London,30379,10
2,E01000003,City of London 001C,E09000001,City of London,14915,5


In [35]:
# Get unique LSOA → force mappings from our crime data
# (clean_df already knows which LSOAs belong to which force)
lsoa_force_map = (
    clean_df[clean_df["lsoa_code"].notna()]
    [["lsoa_code", "force_name"]]
    .drop_duplicates()
)

print(f"Unique LSOA-force mappings from crime data: {len(lsoa_force_map)}")

# Check if any LSOAs map to more than one force (shouldn't happen)
dupes = lsoa_force_map.groupby("lsoa_code")["force_name"].nunique()
multi_force = dupes[dupes > 1]
print(f"LSOAs appearing in more than one force: {len(multi_force)}")

# Join IMD data to LSOA-force mapping
imd_mapped = lsoa_force_map.merge(imd_df[["lsoa_code", "imd_rank", "imd_decile"]], 
                                   on="lsoa_code", how="left")

print(f"\nLSOAs matched to IMD data: {imd_mapped['imd_rank'].notna().sum():,}")
print(f"LSOAs not matched:         {imd_mapped['imd_rank'].isna().sum():,}")

# Aggregate to force level — average IMD decile per force
imd_force = (
    imd_mapped.groupby("force_name")
    .agg(
        imd_avg_decile  = ("imd_decile", "mean"),
        imd_avg_rank    = ("imd_rank",   "mean"),
        lsoa_count      = ("lsoa_code",  "count")
    )
    .round(2)
    .reset_index()
)

print(f"\nForce-level IMD summary:")
print(imd_force)
print(f"\nNote: Lower decile = more deprived (1 = most deprived, 10 = least deprived)")

Unique LSOA-force mappings from crime data: 16478
LSOAs appearing in more than one force: 1110

LSOAs matched to IMD data: 15,243
LSOAs not matched:         1,235

Force-level IMD summary:
              force_name  imd_avg_decile  imd_avg_rank  lsoa_count
0    Metropolitan Police            5.25      15609.97       11642
1   Thames Valley Police            7.48      23040.62        1557
2   West Midlands Police            3.88      11093.28        1803
3  West Yorkshire Police            4.47      13004.69        1476

Note: Lower decile = more deprived (1 = most deprived, 10 = least deprived)
